# S-Learner（CausalML + Optuna）：从 GitHub 克隆并运行

本 notebook **不依赖** 本机 monorepo 固定路径：从 GitHub 拉代码后，在 `MODEL_BUILD` 目录下执行（与 `run_dr_learner_from_github.ipynb` 同源约定）。

- **布局**：`MODEL_BUILD` 下应有 `utils/`、`s_learner_pipeline/`、`requirements_s_learner_pipeline.txt`。若你的仓库根是整仓而非 `model_build`，请设置环境变量 **`LIFT_MODEL_BUILD_REL`**（相对克隆根的路径，例如 `feature_engineering/model_build`）。
- **依赖**：`pip install -r requirements_s_learner_pipeline.txt`。
- **数据库**：`DB_CONFIG` + `set_db_config_inline`；密码用 **`LIFT_MYSQL_PASSWORD`** 等注入，勿提交明文。
- **参数**：**`PIPELINE_CONFIG`** + **`PIPELINE_OVERRIDES`**（含 `learner_families`、`use_sample_weight`、`optuna_n_trials` 等），日常不必读仓库内 YAML。


## 1. 克隆 / 更新仓库


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get(
    "LIFT_GITHUB_REPO_URL",
    "https://github.com/Jialu0901/lift-test-calibration.git",
)
REPO_DIR_NAME = os.environ.get("LIFT_REPO_DIR_NAME", "lift-test-calibration")
BRANCH = os.environ.get("LIFT_REPO_BRANCH", "main")

CLONE_ROOT = Path(os.environ.get("LIFT_CLONE_ROOT", str(Path.home() / "lift_repos"))).expanduser().resolve()
CLONE_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = (CLONE_ROOT / REPO_DIR_NAME).resolve()

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
    print("Updated existing clone:", REPO_DIR)
else:
    if REPO_DIR.exists():
        raise FileExistsError(
            f"{REPO_DIR} exists and is not a git repo; remove it or change REPO_DIR_NAME / LIFT_CLONE_ROOT"
        )
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )
    print("Cloned to:", REPO_DIR)


In [ ]:
import os
import sys
from pathlib import Path

_rel = (os.environ.get("LIFT_MODEL_BUILD_REL") or "").strip().replace("\\", "/")
MODEL_BUILD = (REPO_DIR / _rel).resolve() if _rel else REPO_DIR
if not (MODEL_BUILD / "s_learner_pipeline").is_dir():
    raise FileNotFoundError(
        f"Expected s_learner_pipeline under {MODEL_BUILD}. "
        "Set LIFT_MODEL_BUILD_REL (e.g. feature_engineering/model_build) if the clone root is not model_build."
    )

os.chdir(MODEL_BUILD)
if str(MODEL_BUILD) not in sys.path:
    sys.path.insert(0, str(MODEL_BUILD))

print("cwd:", Path.cwd())
print("MODEL_BUILD:", MODEL_BUILD)


## 2. 安装依赖


In [ ]:
import subprocess
import sys
from pathlib import Path

req = Path.cwd() / "requirements_s_learner_pipeline.txt"
if not req.is_file():
    raise FileNotFoundError(f"Missing {req} — check MODEL_BUILD / LIFT_MODEL_BUILD_REL.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
    check=True,
    cwd=str(MODEL_BUILD),
)
print("pip install OK")


## 3. `DB_CONFIG`（优先环境变量，勿提交密码）


In [ ]:
import os
from pathlib import Path

from utils.db_config import read_db_config_json

_base: dict = {}
try:
    if (os.environ.get("LIFT_DB_CONFIG_JSON") or "").strip():
        _base = read_db_config_json(os.environ["LIFT_DB_CONFIG_JSON"].strip())
    else:
        _base = read_db_config_json()
except FileNotFoundError:
    pass


def _pick_str(env_key: str, base_key: str, default: str) -> str:
    v = (os.environ.get(env_key) or "").strip()
    if v:
        return v
    b = str(_base.get(base_key) or "").strip()
    return b or default


_host = _pick_str("LIFT_MYSQL_HOST", "host", "127.0.0.1")
_user = _pick_str("LIFT_MYSQL_USER", "user", "root")
_database = _pick_str("LIFT_MYSQL_DATABASE", "database", "test")

_port_s = (os.environ.get("LIFT_MYSQL_PORT") or "").strip()
if _port_s:
    _port = int(_port_s)
else:
    _pr = _base.get("port", 3306)
    _port = int(_pr) if _pr is not None else 3306

_pw = (os.environ.get("LIFT_MYSQL_PASSWORD") or "").strip()
_pw_file = (os.environ.get("LIFT_MYSQL_PASSWORD_FILE") or "").strip()
if not _pw and _pw_file:
    _pw = Path(_pw_file).expanduser().read_text(encoding="utf-8").splitlines()[0].strip()
if not _pw:
    _pw = str(_base.get("password") or "")

DB_CONFIG = {
    "host": _host,
    "user": _user,
    "password": _pw,
    "database": _database,
    "port": _port,
    "connection_timeout": 600,
    "consume_results": True,
}

if not DB_CONFIG["password"]:
    print(
        "Hint: set LIFT_MYSQL_PASSWORD, LIFT_MYSQL_PASSWORD_FILE, or LIFT_DB_CONFIG_JSON."
    )
else:
    print("DB_CONFIG ready (host=%r, user=%r, database=%r)" % (_host, _user, _database))


## 3b. `SPLIT_DATES` + `PIPELINE_CONFIG`（全内存）


In [ ]:
SPLIT_DATES = {
    "train": [
        "2025-07-01",
        "2025-07-15",
        "2025-08-01",
        "2025-08-15",
        "2025-09-01",
        "2025-09-15",
        "2025-10-01",
        "2025-10-15",
        "2025-11-01",
        "2025-11-15",
        "2025-11-28",
    ],
    "val": ["2025-12-01", "2025-12-15"],
    "test": ["2025-12-25", "2026-01-01"],
}

PIPELINE_CONFIG: dict = {
    "random_seed": 42,
    "db_table": "workmagic_bi.lift_wide_scaled_v1",
    "wide_table_path": None,
    "sample_limit": None,
    "chunk_days": 1,
    "channels": None,
    "output_root": "output",
    "selected_features_csv": None,
    "use_sample_weight": True,
    "sample_weight_column": "sample_weight",
    "feature_selection_protected_features": [
        "year", "month", "day", "quarter", "day_of_week", "is_weekday",
        "is_us_event", "event_pre_days", "event_post_days",
        "is_hist_order_missing", "is_hist_adview_missing",
    ],
    "feature_selection_protected_prefixes": [
        "day_name_", "us_event_name_", "us_event_type_",
        "emb_user_base_", "emb_event_",
    ],
    "learner_families": ["lgbm"],
    "optuna_n_trials": 15,
    "optuna_timeout": None,
    "refit_on_train_val": False,
    "n_rank_bins": 10,
    "placebo_shuffle_repeats": 20,
    "eval_placebo_on": "val",
}

PIPELINE_OVERRIDES: dict = {}

DB_TABLE: str | None = None


## 4. 两步：加载划分 + 训练


In [ ]:
import copy
import inspect
import json
import tempfile
from pathlib import Path

import yaml

from utils.db_config import set_db_config_inline

import s_learner_pipeline.run_pipeline as _slp

if not hasattr(_slp, "load_wide_and_split"):
    raise RuntimeError(
        f"Clone is missing load_wide_and_split. cd {MODEL_BUILD} && git pull origin main"
    )


def _merge_fallback(base: dict, overrides: dict) -> dict:
    out = copy.deepcopy(base)
    for k, v in (overrides or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _merge_fallback(out[k], v)
        else:
            out[k] = copy.deepcopy(v)
    return out


merge_pipeline_config = getattr(_slp, "merge_pipeline_config", _merge_fallback)
from s_learner_pipeline.run_pipeline import load_wide_and_split

set_db_config_inline(DB_CONFIG)

cfg = merge_pipeline_config(copy.deepcopy(PIPELINE_CONFIG), PIPELINE_OVERRIDES)
if DB_TABLE:
    cfg["db_table"] = DB_TABLE

with tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False, encoding="utf-8") as tf:
    yaml.safe_dump(cfg, tf, sort_keys=False, allow_unicode=True)
    cfg_yaml_path = Path(tf.name)

bundle = load_wide_and_split(cfg, SPLIT_DATES)
print("df shape:", bundle.df.shape, "| load:", bundle.date_load_start, "..", bundle.date_load_end)
print("train", len(bundle.train_df), "val", len(bundle.val_df), "test", len(bundle.test_df))
print("cfg snapshot:", cfg_yaml_path)


In [ ]:
from s_learner_pipeline.run_pipeline import run_pipeline_training_from_splits

cli_overrides: dict = {
    "split_dates_source": "inline_dict",
    "db_table": cfg.get("db_table"),
}

exp_dir = run_pipeline_training_from_splits(
    cfg,
    SPLIT_DATES,
    cfg_yaml_path,
    cli_overrides,
    bundle,
    verbose=True,
)
print("Done:", exp_dir)
